# Project A: bounded Colab smoke run

This notebook calls the same `paper2` package and CLI as Vast. It does not implement another trainer. Start with a **GPU runtime with L4/A100**; the declared profile requires BF16. No pretrained-model or scientific result is implied by CPU test success.

The default Git mode downloads the pinned Project A source directly from GitHub. No chat download or ZIP upload is needed. Keep the recorded source revision for reproducibility; do not paste credentials into this notebook.

Select **Python 3.12** and an **L4 or A100** in Runtime → Change runtime type. In the next cell, authorize this session, enter its incremental cash allowance/rate (both `0.0` when no additional cash is allowed or billed), and confirm your intended Drive destination. Then run the cells in order. The three-update limit is an initial software smoke test.

The first run is limited to three updates. Do not enlarge it until the model preflight, gradients, checkpoint/resume, throughput, and durable export have been checked. Final tests remain unopened.


In [ ]:
# Explicit user choices; existing authorization fields apply to this one bounded session.
SOURCE_MODE = "git"                 # "upload" or "git"
SOURCE_SHA256 = ""                     # Optional expected source ZIP SHA256; always record the actual hash.
GIT_URL = "https://github.com/syedzahedi1990/followuprl.git"
GIT_REVISION = "75c3d857b5ec22d9eeb1d0aad41e05c26ceca718"  # Reviewed source commit; retain for resume.
GITHUB_TOKEN_SECRET = ""                # Private repo: name of a Colab Secret with read-only Contents access.
RUNTIME_AUTHORIZED = False
AUTHORIZED_SPEND_USD = None             # Explicit 0.0 is valid for no incremental cash allowance.
HOURLY_INCREMENTAL_CASH_USD = None      # Explicit 0.0 is valid for existing-subscription cash accounting.
DURABILITY_CONFIRMED = False
DURABLE_SUBDIRECTORY = "paper2-results"
MAX_RUNTIME_MINUTES = 30                # ONE session limit: source/setup/checks/resume/train/export.
INITIAL_COMPUTE_UNITS = None            # Optional manual Colab resource-panel reading.
FINAL_COMPUTE_UNITS = None              # Optional manual reading after work.
RESTORE_BUNDLE = ""                    # Leave blank for a new smoke; otherwise a verified Drive export directory.
EXISTING_NEWER_LEDGER = ""              # Optional verified extension of the exported accounting history.


In [ ]:
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import sys
import time
import uuid
from pathlib import Path

if not RUNTIME_AUTHORIZED:
    raise RuntimeError("Authorize this specific Colab session before installing or running model work.")
if AUTHORIZED_SPEND_USD is None or HOURLY_INCREMENTAL_CASH_USD is None:
    raise RuntimeError("Set explicit incremental cash allowance/rate, including zeros if applicable.")
if not (0 < MAX_RUNTIME_MINUTES <= 30):
    raise RuntimeError("Keep the initial session cap at no more than 30 minutes.")
SESSION_STARTED = time.time()
SESSION_CLOCK_STARTED = time.monotonic()
SESSION_DEADLINE = SESSION_CLOCK_STARTED + MAX_RUNTIME_MINUTES * 60
EXPORT_RESERVE_SECONDS = 90
STOP_GRACE_SECONDS = 15
WORK_DEADLINE = SESSION_DEADLINE - EXPORT_RESERVE_SECONDS

def remaining_seconds(*, cleanup=False):
    return max(0.0, (SESSION_DEADLINE if cleanup else WORK_DEADLINE) - time.monotonic())

def run_bounded(command, *, cwd=None, capture_output=False, cleanup=False, check=True, env=None):
    remaining = remaining_seconds(cleanup=cleanup)
    if remaining <= STOP_GRACE_SECONDS:
        raise TimeoutError("Session time is exhausted; no new work will start. Export known completed results.")
    process = subprocess.Popen(command, cwd=cwd, start_new_session=True, text=True, env=env,
                               stdout=subprocess.PIPE if capture_output else None,
                               stderr=subprocess.PIPE if capture_output else None)
    try:
        out, err = process.communicate(timeout=remaining-STOP_GRACE_SECONDS)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            out, err = process.communicate(timeout=max(0.1, remaining_seconds(cleanup=cleanup)))
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            out, err = process.communicate()
        error = TimeoutError("The shared session deadline stopped this process group; inspect/export complete checkpoints.")
        error.stdout, error.stderr = out, err
        raise error
    result = subprocess.CompletedProcess(command, process.returncode, out, err)
    if check:
        result.check_returncode()
    return result

if platform.python_version_tuple()[:2] != ("3", "12"):
    raise RuntimeError("Select Python 3.12 for this locked profile; do not silently change the lock.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU is attached. Select a GPU runtime.")
run_bounded(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
print("Python:", platform.python_version())
print("Local free GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))
print("Total session cap, including installation/checks:", MAX_RUNTIME_MINUTES, "minutes")


## Import exact source

Git mode uses the source commit already filled in above and records the verified checkout SHA. The optional upload path accepts a repository ZIP and records its checksum.

For a private repository, authorize Colab to open the notebook, then add a fine-grained GitHub token to the Colab **Secrets** panel (key icon). Limit the token to this repository with **Contents: Read-only**, enable notebook access, and put only the secret name in `GITHUB_TOKEN_SECRET` above. Browser GitHub authentication alone does not authenticate runtime Git. The token is passed only to Git through an environment variable and a temporary askpass helper; it is not placed in URLs, saved Git config, or notebook output.

Alternatively, download **Code → Download ZIP** from GitHub, set `SOURCE_MODE = "upload"`, and upload that ZIP here. Neither route requires a chat download or changes repository visibility.


In [ ]:
import stat
import zipfile

source_area = Path("/content/paper2-source")
source_area.mkdir(exist_ok=True)
if SOURCE_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    archives = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise RuntimeError("Upload exactly one source ZIP.")
    archive_name, payload = archives[0]
    source_hash = hashlib.sha256(payload).hexdigest()
    if SOURCE_SHA256 and source_hash != SOURCE_SHA256.lower():
        raise RuntimeError("Source ZIP checksum mismatch.")
    archive_path = Path("/content/paper2-source.zip")
    archive_path.write_bytes(payload)
    checkout = source_area / source_hash[:16]
    if not checkout.exists():
        checkout.mkdir()
        with zipfile.ZipFile(archive_path) as archive:
            if len(archive.infolist()) > 10000 or sum(m.file_size for m in archive.infolist()) > 2 * 2**30:
                raise RuntimeError("Unexpectedly large source archive; use a source-only bundle without weights.")
            for member in archive.infolist():
                target = (checkout / member.filename).resolve()
                if not target.is_relative_to(checkout.resolve()):
                    raise RuntimeError("Unsafe archive path.")
                if stat.S_ISLNK(member.external_attr >> 16):
                    raise RuntimeError("Symlinks are not accepted in the source archive.")
            archive.extractall(checkout)
    candidates = list(checkout.rglob("pyproject.toml"))
    if len(candidates) != 1:
        raise RuntimeError("Source archive must contain exactly one project pyproject.toml.")
    REPO = candidates[0].parent
    source_identity = {"kind":"uploaded_source_zip", "sha256":source_hash}
elif SOURCE_MODE == "git":
    if not re.fullmatch(r"[0-9a-fA-F]{40}", GIT_REVISION):
        raise RuntimeError("Set an exact full Git commit SHA, not a branch name.")
    REPO = source_area / GIT_REVISION.lower()
    if not REPO.exists():
        git_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        askpass = None
        try:
            if GITHUB_TOKEN_SECRET:
                from urllib.parse import urlparse
                from google.colab import userdata
                parsed = urlparse(GIT_URL)
                if (parsed.scheme, parsed.hostname) != ("https", "github.com") or parsed.username or parsed.password:
                    raise ValueError("A GitHub Secret may only authenticate an HTTPS github.com URL without embedded credentials.")
                git_env["PAPER2_GITHUB_TOKEN"] = userdata.get(GITHUB_TOKEN_SECRET)
                askpass = source_area / f".git-askpass-{uuid.uuid4().hex}.py"
                askpass.write_text(
                    "#!/usr/bin/env python3\nimport os, sys\n"
                    "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['PAPER2_GITHUB_TOKEN'])\n"
                )
                askpass.chmod(0o700)
                git_env["GIT_ASKPASS"] = str(askpass)
            run_bounded(["git", "-c", "credential.helper=", "clone", GIT_URL, str(REPO)], env=git_env)
        finally:
            git_env.pop("PAPER2_GITHUB_TOKEN", None)
            if askpass is not None:
                askpass.unlink(missing_ok=True)
    run_bounded(["git", "checkout", "--detach", GIT_REVISION], cwd=REPO)
    actual_revision = run_bounded(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True).stdout.strip()
    if actual_revision.lower() != GIT_REVISION.lower():
        raise RuntimeError("Checkout revision mismatch.")
    source_identity = {"kind":"git_commit", "commit":actual_revision}
else:
    raise RuntimeError("SOURCE_MODE must be upload or git.")
print("Source:", source_identity)
print("Repository:", REPO)


## Mount durable storage and install the lock

Train on runtime-local disk and export completed bundles to Drive. Merely creating a directory does not establish durability: confirm it belongs to the intended Drive and is accessible from a separate session. Colab runtime duration and hardware are not guaranteed.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DURABLE = Path("/content/drive/MyDrive") / DURABLE_SUBDIRECTORY
if not DURABLE.resolve().is_relative_to(Path("/content/drive/MyDrive").resolve()):
    raise RuntimeError("Durable subdirectory must remain under MyDrive.")
DURABLE.mkdir(parents=True, exist_ok=True)
probe = DURABLE / ".paper2-write-check"
probe.write_text("write-check", encoding="utf-8")
if probe.read_text(encoding="utf-8") != "write-check":
    raise RuntimeError("Durable write/read check failed.")
probe.unlink()
if not DURABILITY_CONFIRMED:
    raise RuntimeError("Verify this destination is on your intended durable Drive, then set DURABILITY_CONFIRMED=True.")
VENV = Path("/content/paper2-venv")
os.environ["HF_HOME"] = "/content/paper2-hf-cache"
run_bounded(["bash", "scripts/bootstrap_vast.sh", "--gpu", "--python", sys.executable, "--venv", str(VENV)], cwd=REPO)
PYTHON = str(VENV / "bin/python")


## Resolve the runtime configuration or restore a completed export

The same authorization fields apply throughout this session. Existing-subscription incremental cash may be explicitly zero; elapsed time and available compute units remain recorded. The shared deadline includes installation and all checks. If it expires, export known completed results and stop.

For interruption recovery, set `RESTORE_BUNDLE` before running the notebook. The package verifies and restores the bundle into a new local directory, preserving checkpoint and accounting history. The configuration is then updated with this session's explicitly selected runtime/durable destination. Use the same source, model, software, and numerical backend; changed GPU hardware requires separate comparability validation.


In [ ]:
local_inputs = {
    "runtime_authorized": RUNTIME_AUTHORIZED,
    "authorized_spend_usd": float(AUTHORIZED_SPEND_USD),
    "hourly_rate_usd": float(HOURLY_INCREMENTAL_CASH_USD),
    "durable_destination": str(DURABLE),
    "durability_confirmed": DURABILITY_CONFIRMED,
    "job_walltime_cap_minutes": MAX_RUNTIME_MINUTES,
}
RESTORED = None
base_profile = REPO / "configs/gpu_smoke.yaml"
if RESTORE_BUNDLE:
    restore_destination = Path("/content") / f"paper2-restored-{uuid.uuid4().hex[:8]}"
    command = [PYTHON, "-m", "paper2", "restore", "--bundle", RESTORE_BUNDLE,
               "--destination", str(restore_destination)]
    if EXISTING_NEWER_LEDGER:
        command += ["--ledger-path", EXISTING_NEWER_LEDGER]
    restored_process = run_bounded(command, cwd=REPO, capture_output=True)
    RESTORED = json.loads(restored_process.stdout)
    base_profile = Path(RESTORED["config_path"])
    print("Restored verified bundle:", RESTORED)
CONFIG = REPO / f"configs/local_gpu_{uuid.uuid4().hex[:8]}.yaml"
config_script = r"""
import json, pathlib, sys, yaml
root, profile, destination = map(pathlib.Path, sys.argv[1:4])
settings, restoring = json.loads(sys.argv[4]), sys.argv[5] == 'true'
cfg = yaml.safe_load(profile.read_text())
cfg.setdefault('budget', {}).update(settings)
cfg['budget']['mode'] = 'colab'
if not restoring:
    cfg['budget']['stage'] = 'validation'
    cfg['budget']['stage_caps_usd'] = {'validation':50, 'pilot':100, 'recovery':200, 'confirmation':100, 'reserve':50}
    cfg['budget']['ledger_path'] = str(root / 'runs/budget.jsonl')
    cfg['output_dir'] = str(root / 'runs/colab_smoke')
    cfg.setdefault('training', {}).update(phase_update_cap=3, evaluation_every_updates=1, unique_prompts_per_update=8, microbatch_size=1)
    cfg.setdefault('checkpoint', {})['save_every_updates'] = 1
with destination.open('x') as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)
"""
run_bounded([PYTHON, "-c", config_script, str(REPO), str(base_profile), str(CONFIG),
             json.dumps(local_inputs), str(bool(RESTORED)).lower()])
(REPO / "runs").mkdir(exist_ok=True)
(REPO / "runs/colab_source_identity.json").write_text(json.dumps(source_identity, indent=2))
print("Local configuration:", CONFIG)
print("Work minutes remaining:", round(remaining_seconds()/60, 2))


In [ ]:
LAUNCHER_LOGS = REPO / "runs/launcher_logs" / str(int(SESSION_STARTED))
LAUNCHER_LOGS.mkdir(parents=True, exist_ok=True)
command_counter = 0

def paper2(*args, capture_output=False, cleanup=False):
    global command_counter
    args = list(map(str, args))
    if args[0] in {"train", "preflight", "resume-check"}:
        config_path = args[args.index("--config") + 1]
        # Read this file's existing cap rather than enlarging the CPU profile's shorter cap.
        cap_query = "import sys,yaml; print(yaml.safe_load(open(sys.argv[1]))['budget']['job_walltime_cap_minutes'])"
        configured_cap = float(run_bounded([PYTHON, "-c", cap_query, config_path], cwd=REPO, capture_output=True).stdout)
        remaining = remaining_seconds() - STOP_GRACE_SECONDS
        if remaining <= 0:
            raise TimeoutError("No work time remains; only export will be attempted.")
        args += ["--walltime-minutes", f"{min(configured_cap, remaining/60):.6f}"]
    command_counter += 1
    log_path = LAUNCHER_LOGS / f"{command_counter:03d}-{args[0]}.log"
    try:
        result = run_bounded([PYTHON, "-m", "paper2", *args], cwd=REPO,
                             capture_output=True, cleanup=cleanup)
    except Exception as exc:
        log_path.write_text(str(exc) + "\n" + (getattr(exc, "stdout", None) or "") + "\n" + (getattr(exc, "stderr", None) or ""))
        raise
    log_path.write_text(result.stdout + "\nSTDERR:\n" + result.stderr)
    if not capture_output:
        print(result.stdout)
        if result.stderr:
            print(result.stderr, file=sys.stderr)
    return result


## Run checks, real-model resume validation, then a bounded job

This single cell uses the remaining **session** time for CPU checks, real-model preflight, fresh-process real-model resume validation, and training. Every subprocess has the same shrinking deadline; it never receives a fresh 30-minute allowance. Ninety seconds are reserved for export. No new work starts after the work budget expires.

For a new run, the corrected-verifier `clean` phase is only a three-update software smoke. For a restored run, the original phase, seed, arm, schedule, and checkpoint are retained. The real-model GPU resume path is implemented but has not been executed in the CPU-only development environment.


In [ ]:
before_manifests = set((REPO / "runs").rglob("*manifest*.json"))
job_started = time.time()
RUN_MANIFESTS = []
job_error = None
try:
    if not RESTORED:
        RESOLVED_CONFIG = CONFIG.with_name(CONFIG.stem + ".resolved.yaml")
        paper2("resolve-model", "--config", CONFIG, "--output", RESOLVED_CONFIG)
        CONFIG = RESOLVED_CONFIG
    paper2("prepare", "--config", CONFIG)
    paper2("validate", "--config", CONFIG, "--cpu-only")
    # CPU engineering check: cannot be mistaken for the following real-model check.
    paper2("resume-check", "--config", "configs/resume_smoke.yaml")
    paper2("preflight", "--config", CONFIG)
    paper2("resume-check", "--config", CONFIG)
    if RESTORED:
        restored_manifest = json.loads(Path(RESTORED["manifest_path"]).read_text())
        paper2("train", "--config", CONFIG, "--phase", restored_manifest["phase"],
               "--seed", restored_manifest["seed"], "--arm", restored_manifest["arm"],
               "--resume", RESTORED["final_checkpoint"])
    else:
        paper2("train", "--config", CONFIG, "--phase", "clean", "--seed", "101")
except Exception as exc:
    job_error = exc
finally:
    usage = {
        "source":source_identity,
        "backend":"colab",
        "session_elapsed_seconds":time.monotonic()-SESSION_CLOCK_STARTED,
        "job_elapsed_seconds":time.time()-job_started,
        "session_cap_minutes":MAX_RUNTIME_MINUTES,
        "initial_compute_units_user_reported":INITIAL_COMPUTE_UNITS,
        "final_compute_units_user_reported":FINAL_COMPUTE_UNITS,
        "incremental_cash_hourly_rate_usd":HOURLY_INCREMENTAL_CASH_USD,
        "error":str(job_error) if job_error else None,
        "note":"Includes source/setup/checks in session time. Compute units are manual observations. Provider billing can include time outside this notebook.",
    }
    usage_file = REPO / "runs/colab_usage.json"
    usage_file.write_text(json.dumps(usage, indent=2))
    shutil.copy2(usage_file, DURABLE / f"colab_usage_{int(SESSION_STARTED)}.json")
    shutil.copy2(REPO / "runs/colab_source_identity.json", DURABLE / f"colab_source_{int(SESSION_STARTED)}.json")
    candidates = sorted(set((REPO / "runs").rglob("*manifest*.json")) - before_manifests)
    if RESTORED:
        candidates.append(Path(RESTORED["manifest_path"]))
    for path in candidates:
        record = json.loads(path.read_text())
        if isinstance(record, dict) and "run_id" in record and record.get("checkpoints"):
            RUN_MANIFESTS.append(path)
    export_errors = []
    for manifest in RUN_MANIFESTS:
        try:
            paper2("export", "--manifest", manifest, "--destination", DURABLE, cleanup=True)
        except Exception as exc:
            export_errors.append({"manifest":str(manifest), "error":str(exc)})
    # Preserve raw check results even if preflight/resume stopped before a training manifest existed.
    import zipfile
    diagnostics_zip = REPO / "runs" / f"colab_diagnostics_{int(SESSION_STARTED)}.zip"
    with zipfile.ZipFile(diagnostics_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for file in sorted(LAUNCHER_LOGS.glob("*.log")):
            archive.write(file, "launcher_logs/" + file.name)
        for file in sorted((REPO / "runs").rglob("result.json")):
            if file.stat().st_mtime >= SESSION_STARTED:
                archive.write(file, str(file.relative_to(REPO / "runs")))
    shutil.copy2(diagnostics_zip, DURABLE / diagnostics_zip.name)
    print("Run manifests:", [str(p) for p in RUN_MANIFESTS])
    print("Session elapsed seconds:", time.monotonic()-SESSION_CLOCK_STARTED)
    print("Export failures requiring attention:", export_errors)
if job_error:
    raise job_error
if export_errors:
    raise RuntimeError("Some exports failed. Preserve the local runtime and inspect before deleting it.")
if not RUN_MANIFESTS:
    raise RuntimeError("No run manifest was produced; inspect the saved check results before starting further work.")


## Inspect and stop

Review the manifest, timing/memory measurements, gradients, optimizer movement, and verified exports. The real-model `resume-check` compares uninterrupted and fresh-process resumed training on the selected backend. This GPU check was **unrun** in the CPU-only development environment; passing the tiny-policy check alone does not validate it.

After interruption, mount the same durable storage and set `RESTORE_BUNDLE` to its completed export directory. The notebook calls `paper2 restore --bundle ... --destination ...`, then applies the existing session authorization fields to the returned configuration. A newer ledger can be supplied only if it extends the exported history. Reconcile any charges incurred after the export before continuing; restoring model state must never erase accounting history.

The restored config, phase, seed, arm, sampler, and full checkpoint are used for resume. An inference-only adapter is insufficient. Inspect newly assigned hardware: if it changes, stratify or repeat the affected GPU-time comparison. No keepalive trick or provider-limit workaround is included.

After the bounded job and successful export verification, disconnect/delete the Colab runtime through its normal UI. If export fails, preserve local data until a verified durable copy exists. The notebook's process deadline cannot guarantee a billing stop after browser disconnect or runtime failure.

Keep final test suites unopened until a confirmatory protocol and exact checkpoints are frozen. Send the smoke manifest, check results, and export checksum index for the next decision; do not share credentials or large model weights in chat.
